# Week 6 Assignment
## Spark Architecture and Efficient Data Processing using PySpark

### Name: Khushi Arora
### Internship:Celebal Technologies Internship
### Week: 6

Objective:
Understand Spark Architecture, Lazy Evaluation, DataFrame Transformations,
Filtering, Schema Handling, CSV vs Parquet,
Predicate Pushdown and Data Pipelines.

In [5]:
# Install PySpark library in Google Colab
!pip install pyspark

In [3]:
# Import SparkSession from PySpark
from pyspark.sql import SparkSession

# Create a Spark Session
spark = SparkSession.builder \
    .appName("Week 6 Spark Assignment") \
    .getOrCreate()

# Print confirmation message
print("Spark Session Created Successfully!")

Spark Session Created Successfully!


In [4]:
# Import upload utility
from google.colab import files

# Upload CSV file from local computer
uploaded = files.upload()

Saving source.csv to source.csv


# Question 1: Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application.

## Objective

Understand the core components of Spark Architecture and how they work together to execute a Spark application efficiently.

---

## Driver

The **Driver** is the main control unit of a Spark application. It is responsible for creating the Spark Session, converting the user program into tasks, scheduling those tasks, and collecting the final results from the Executors.

### Responsibilities:

* Creates the Spark Session.
* Converts user code into execution stages.
* Schedules tasks for execution.
* Monitors job execution.
* Collects the final output from Executors.

---

## Cluster Manager

The **Cluster Manager** is responsible for allocating resources such as CPU cores and memory to the Spark application. It launches Executors on worker nodes and manages cluster resources efficiently.

### Responsibilities:

* Allocates cluster resources.
* Starts Executor processes.
* Manages CPU and memory allocation.
* Supports multiple Spark applications.

**Examples:** Standalone Cluster Manager, Hadoop YARN, Kubernetes, Apache Mesos.

---

## Executor

An **Executor** is a worker process responsible for executing the tasks assigned by the Driver. Executors process data, perform transformations and actions, cache intermediate data when required, and send the results back to the Driver.

### Responsibilities:

* Executes assigned tasks.
* Processes data partitions.
* Stores cached data in memory.
* Returns results to the Driver.

---

## Spark Architecture

```text
                   User Program
                        │
                        ▼
                 +---------------+
                 |    Driver     |
                 +---------------+
                        │
          Requests Cluster Resources
                        │
                        ▼
              +----------------------+
              |  Cluster Manager     |
              +----------------------+
                  │              │
                  ▼              ▼
         +----------------+ +----------------+
         |   Executor 1   | |   Executor 2   |
         +----------------+ +----------------+
                  │              │
            Process Data    Process Data
                  │              │
                  └──────► Results ◄──────┘
                         Returned to Driver

---

## Conclusion

The **Driver** acts as the brain of the Spark application, the **Cluster Manager** allocates system resources, and the **Executors** perform the actual data processing. Together, these components enable Spark to process large datasets efficiently in a distributed environment.
```


# Question 2: How does Spark’s Lazy Evaluation strategy improve performance when chain-processing large datasets?

## Answer

**Lazy Evaluation** is a feature of Apache Spark in which transformations are **not executed immediately**. Instead, Spark records all transformations and waits until an **Action** (such as `show()`, `count()`, or `write()`) is called.

Before execution, Spark creates a **Directed Acyclic Graph (DAG)** that stores the sequence of transformations. It then analyzes this DAG to optimize the execution plan by combining operations, removing unnecessary computations, and reducing data movement.

### Advantages of Lazy Evaluation

* Improves execution performance.
* Reduces unnecessary computations.
* Optimizes the execution plan using DAG.
* Minimizes disk I/O and network communication.
* Makes processing of large datasets more efficient.

**Conclusion:**
Lazy Evaluation allows Spark to execute multiple transformations together in an optimized manner, resulting in faster and more efficient processing of large datasets.



In [6]:
# ============================================================
# Question 3: Read CSV File with Header and Infer Schema
# ============================================================

# Read the uploaded CSV file into a Spark DataFrame
df = spark.read.csv(
    "source.csv",      # CSV file name
    header=True,       # Use the first row as column names
    inferSchema=True   # Automatically detect data types
)

# Display the first five rows of the DataFrame
df.show(5)

# Display the DataFrame schema
df.printSchema()

+----------+-----------+-----+--------+---------+------+----------+------+--------+-------+
|product_id|   category|price|old_name|   status|amount|base_price|region|priority|user_id|
+----------+-----------+-----+--------+---------+------+----------+------+--------+-------+
|       101|Electronics|25000|  Laptop|Completed| 25000|     25000| North|    High|   1001|
|       102|  Furniture| 7000|   Chair|  Pending|  7000|      7000| South|     Low|   1002|
|       103|Electronics|12000|   Phone|Completed| 12000|     12000|  East|  Medium|   1003|
|       104|Electronics|  900|   Mouse|Completed|   900|       900| North|    High|   NULL|
|       105|   Clothing| 2000|   Shirt|Cancelled|  2000|      2000|  West|     Low|   1005|
+----------+-----------+-----+--------+---------+------+----------+------+--------+-------+
only showing top 5 rows
root
 |-- product_id: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- old_name: string 

# Question 4: What is the difference between CSV and Parquet in terms of storage (row-based vs. columnar) and why does it matter for performance?

## Answer

| CSV                                        | Parquet                                                        |
| ------------------------------------------ | -------------------------------------------------------------- |
| CSV is a **row-based** storage format.     | Parquet is a **columnar** storage format.                      |
| Stores data row by row.                    | Stores data column by column.                                  |
| Requires more storage space.               | Uses compression, so it requires less storage space.           |
| Slower to read and process large datasets. | Faster to read because only the required columns are accessed. |
| Does not support Predicate Pushdown.       | Supports Predicate Pushdown for better query performance.      |

### Why does it matter for performance?

Parquet improves Spark performance because it reads only the required columns instead of the entire dataset. It also uses compression and supports Predicate Pushdown, which reduces disk I/O and memory usage. Therefore, Parquet is the preferred file format for big data processing in Apache Spark.


In [7]:
# ============================================================
# Question 5: Select product_id and price where category is 'Electronics'
# ============================================================

# Filter rows where the category is 'Electronics'
electronics_df = df.filter(df.category == "Electronics")

# Select only the required columns
electronics_df = electronics_df.select("product_id", "price")

# Display the filtered DataFrame
electronics_df.show()

+----------+-----+
|product_id|price|
+----------+-----+
|       101|25000|
|       103|12000|
|       104|  900|
|       106|18000|
|       108|35000|
+----------+-----+



In [8]:
# ============================================================
# Question 6: Rename a Column and Cast Data Type
# ============================================================

# Import the col() function
from pyspark.sql.functions import col

# Rename the column 'old_name' to 'new_name'
df = df.withColumnRenamed("old_name", "new_name")

# Convert the 'price' column to Double data type
df = df.withColumn("price", col("price").cast("double"))

# Display the updated DataFrame
df.show(5)

# Display the updated schema
df.printSchema()

+----------+-----------+-------+--------+---------+------+----------+------+--------+-------+
|product_id|   category|  price|new_name|   status|amount|base_price|region|priority|user_id|
+----------+-----------+-------+--------+---------+------+----------+------+--------+-------+
|       101|Electronics|25000.0|  Laptop|Completed| 25000|     25000| North|    High|   1001|
|       102|  Furniture| 7000.0|   Chair|  Pending|  7000|      7000| South|     Low|   1002|
|       103|Electronics|12000.0|   Phone|Completed| 12000|     12000|  East|  Medium|   1003|
|       104|Electronics|  900.0|   Mouse|Completed|   900|       900| North|    High|   NULL|
|       105|   Clothing| 2000.0|   Shirt|Cancelled|  2000|      2000|  West|     Low|   1005|
+----------+-----------+-------+--------+---------+------+----------+------+--------+-------+
only showing top 5 rows
root
 |-- product_id: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- 

# Question 7: How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails?

## Answer

Apache Spark maintains a **Lineage Graph**, also known as a **Directed Acyclic Graph (DAG)**, which records all the transformations applied to a DataFrame or RDD. Instead of storing multiple copies of intermediate data, Spark keeps track of the sequence of operations.

If a worker node or executor fails, Spark uses the DAG to identify the lost data partition and automatically recomputes only that partition from the original data and recorded transformations. This eliminates the need to restart the entire job, providing efficient fault tolerance and reliable execution of large-scale data processing tasks.

**Conclusion:**
The Lineage Graph (DAG) enables Spark to recover lost data efficiently by recomputing only the affected partitions, ensuring fault tolerance without data replication.


In [13]:
# ============================================================
# Question 8: Filter Completed Orders with Amount > 1000
# ============================================================

# Create a reference DataFrame named df_orders
df_orders = df

# Filter completed orders with amount greater than 1000
completed_orders = df_orders.filter(
    (df_orders.status == "Completed") &
    (df_orders.amount > 1000)
)

# Display the filtered DataFrame
completed_orders.show()

+----------+-----------+-------+--------+---------+------+----------+------+--------+-------+-----------+
|product_id|   category|  price|new_name|   status|amount|base_price|region|priority|user_id|final_price|
+----------+-----------+-------+--------+---------+------+----------+------+--------+-------+-----------+
|       101|Electronics|25000.0|  Laptop|Completed| 25000|     25000| North|    High|   1001|    29500.0|
|       103|Electronics|12000.0|   Phone|Completed| 12000|     12000|  East|  Medium|   1003|    14160.0|
|       106|Electronics|18000.0|  Tablet|Completed| 18000|     18000| North|  Medium|   1006|    21240.0|
|       107|  Furniture| 4500.0|   Table|Completed|  4500|      4500| South|    High|   1007|     5310.0|
+----------+-----------+-------+--------+---------+------+----------+------+--------+-------+-----------+



# Question 9: Explain the concept of Predicate Pushdown in Parquet and how it affects the amount of data loaded into memory.

## Answer

**Predicate Pushdown** is an optimization technique supported by the Parquet file format. When a filter condition is applied, Spark pushes the filter down to the Parquet file instead of loading the entire dataset into memory.

As a result, only the rows and columns that satisfy the filter condition are read from the disk. This significantly reduces disk I/O, memory usage, and query execution time.

**Conclusion:**
Predicate Pushdown improves Spark performance by reading only the required data from Parquet files, making data processing faster and more efficient.


In [10]:
# ============================================================
# Question 10: Add a New Column 'final_price'
# ============================================================

# Import the col() function
from pyspark.sql.functions import col

# Create a new column 'final_price' by adding 18% tax
df = df.withColumn(
    "final_price",
    col("base_price") * 1.18
)

# Display the updated DataFrame
df.show(5)

+----------+-----------+-------+--------+---------+------+----------+------+--------+-------+-----------+
|product_id|   category|  price|new_name|   status|amount|base_price|region|priority|user_id|final_price|
+----------+-----------+-------+--------+---------+------+----------+------+--------+-------+-----------+
|       101|Electronics|25000.0|  Laptop|Completed| 25000|     25000| North|    High|   1001|    29500.0|
|       102|  Furniture| 7000.0|   Chair|  Pending|  7000|      7000| South|     Low|   1002|     8260.0|
|       103|Electronics|12000.0|   Phone|Completed| 12000|     12000|  East|  Medium|   1003|    14160.0|
|       104|Electronics|  900.0|   Mouse|Completed|   900|       900| North|    High|   NULL|     1062.0|
|       105|   Clothing| 2000.0|   Shirt|Cancelled|  2000|      2000|  West|     Low|   1005|     2360.0|
+----------+-----------+-------+--------+---------+------+----------+------+--------+-------+-----------+
only showing top 5 rows


# Question 11: What is the difference between Transformations and Actions? Provide two examples of each.

## Answer

In Apache Spark, **Transformations** create a new DataFrame or RDD from an existing one without executing immediately. They are lazily evaluated and are executed only when an **Action** is called.

**Examples of Transformations:**

* `filter()`
* `select()`

**Actions** trigger the execution of all pending transformations and return a result or write data to storage.

**Examples of Actions:**

* `show()`
* `count()`

**Conclusion:**
Transformations define the operations to be performed, while Actions execute those operations and produce the final result.


In [11]:
# ============================================================
# Question 12: Read Parquet, Filter Null Values and Save as CSV
# ============================================================

# Save the current DataFrame as a Parquet file
df.write.mode("overwrite").parquet("sample_parquet")

# Read the Parquet file
parquet_df = spark.read.parquet("sample_parquet")

# Remove rows where 'user_id' is NULL
filtered_df = parquet_df.filter(parquet_df.user_id.isNotNull())

# Save the filtered data as a CSV file
filtered_df.write.mode("overwrite").option("header", True).csv("output_csv")

# Display the filtered DataFrame
filtered_df.show(5)

+----------+-----------+-------+--------+---------+------+----------+------+--------+-------+-----------+
|product_id|   category|  price|new_name|   status|amount|base_price|region|priority|user_id|final_price|
+----------+-----------+-------+--------+---------+------+----------+------+--------+-------+-----------+
|       101|Electronics|25000.0|  Laptop|Completed| 25000|     25000| North|    High|   1001|    29500.0|
|       102|  Furniture| 7000.0|   Chair|  Pending|  7000|      7000| South|     Low|   1002|     8260.0|
|       103|Electronics|12000.0|   Phone|Completed| 12000|     12000|  East|  Medium|   1003|    14160.0|
|       105|   Clothing| 2000.0|   Shirt|Cancelled|  2000|      2000|  West|     Low|   1005|     2360.0|
|       106|Electronics|18000.0|  Tablet|Completed| 18000|     18000| North|  Medium|   1006|    21240.0|
+----------+-----------+-------+--------+---------+------+----------+------+--------+-------+-----------+
only showing top 5 rows


# Question 13: In Spark Architecture, what is the difference between Client Mode and Cluster Mode?

## Answer

| Client Mode                                                  | Cluster Mode                                                      |
| ------------------------------------------------------------ | ----------------------------------------------------------------- |
| The Driver program runs on the local machine.                | The Driver program runs inside the cluster.                       |
| Suitable for development and testing.                        | Suitable for production environments.                             |
| If the client machine disconnects, the application may stop. | The application continues running even if the client disconnects. |
| Easier to debug.                                             | Better for large-scale distributed processing.                    |

**Conclusion:**
Client Mode is mainly used during development, whereas Cluster Mode is preferred for production because it provides better reliability and scalability.


In [12]:
# ============================================================
# Question 14: Filter Region = 'North' OR Priority = 'High'
# ============================================================

# Filter rows where region is 'North' OR priority is 'High'
filtered_region = df.filter(
    (df.region == "North") | (df.priority == "High")
)

# Display the filtered DataFrame
filtered_region.show()

+----------+-----------+-------+--------+---------+------+----------+------+--------+-------+-----------+
|product_id|   category|  price|new_name|   status|amount|base_price|region|priority|user_id|final_price|
+----------+-----------+-------+--------+---------+------+----------+------+--------+-------+-----------+
|       101|Electronics|25000.0|  Laptop|Completed| 25000|     25000| North|    High|   1001|    29500.0|
|       104|Electronics|  900.0|   Mouse|Completed|   900|       900| North|    High|   NULL|     1062.0|
|       106|Electronics|18000.0|  Tablet|Completed| 18000|     18000| North|  Medium|   1006|    21240.0|
|       107|  Furniture| 4500.0|   Table|Completed|  4500|      4500| South|    High|   1007|     5310.0|
|       108|Electronics|35000.0|      TV|  Pending| 35000|     35000|  East|    High|   1008|    41300.0|
+----------+-----------+-------+--------+---------+------+----------+------+--------+-------+-----------+



# Question 15: When exploring a dataset, why is it safer to use `.show(5)` instead of `.collect()` on a multi-terabyte dataset?

## Answer

The `.show(5)` function displays only the first five rows of a DataFrame, making it efficient for exploring large datasets without consuming much memory.

On the other hand, `.collect()` retrieves **all** the data from the cluster and stores it in the Driver's memory. For very large datasets, this can cause excessive memory usage or even lead to an **OutOfMemoryError**.

**Conclusion:**
For large datasets, `.show(5)` is the safer and recommended choice because it displays only a small sample of the data, while `.collect()` should be used only when the dataset is small enough to fit into the Driver's memory.


# Additional Concept: Shuffle in Apache Spark

Shuffle is the process of redistributing data across different partitions during operations such as `groupBy()`, `join()`, and `reduceByKey()`.

Since data moves between executors, shuffle increases network communication and disk I/O, making it one of the most expensive operations in Spark.

To improve performance, Spark applications should minimize unnecessary shuffle operations whenever possible.

# Final Insights

## Performance Insights
- Spark improves performance through Lazy Evaluation and DAG optimization.
- Parquet provides better performance than CSV due to its columnar storage and Predicate Pushdown.
- Filtering data before processing reduces unnecessary computations.
- Using `show()` instead of `collect()` prevents excessive memory usage on large datasets.
- Minimizing shuffle operations improves overall execution efficiency.

## Architecture Insights
- The Driver coordinates the Spark application.
- The Cluster Manager allocates resources.
- Executors process the assigned tasks.
- The Lineage Graph (DAG) enables Spark to recover lost partitions and provides fault tolerance.

## Key Learnings

* Spark follows a distributed computing architecture using Driver, Cluster Manager, and Executors.
* Lazy Evaluation delays execution until an Action is called, allowing Spark to optimize execution.
* The Directed Acyclic Graph (DAG) helps Spark provide fault tolerance and optimize task execution.
* Parquet is more efficient than CSV because it stores data in a columnar format and supports Predicate Pushdown.
* DataFrames provide an efficient way to process structured data.
* Schema inference automatically detects appropriate data types.
* Handling null values improves data quality.
* Transformations are lazily evaluated, while Actions trigger execution.
* Using `show()` is safer than `collect()` when working with large datasets.